# Phase 5 - SVM Classifier Training

In [9]:
# Setting the directories

import os
import json
import pickle
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

LABEL_ORDER = ['Non-Demented', 'Very Mild Demented', 'Mild Demented', 'Moderate Demented']

_cwd = os.path.abspath(os.getcwd())
BASE_DIR = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'notebooks' else _cwd

PROCESSED_DIR = os.path.join(BASE_DIR, 'data', 'processed')
MODELS_DIR    = os.path.join(BASE_DIR, 'outputs', 'models')
METRICS_DIR   = os.path.join(BASE_DIR, 'outputs', 'metrics')

In [10]:
# Loading training data and class weights

X_train = np.load(os.path.join(PROCESSED_DIR, 'X_train.npy'))
y_train = np.load(os.path.join(PROCESSED_DIR, 'y_train.npy'))

with open(os.path.join(METRICS_DIR, 'class_weights.json')) as f:
    class_weight_dict = json.load(f)

print(f'X_train : {X_train.shape}')
print(f'y_train : {y_train.shape}')
print(f'Class weights: {class_weight_dict}')


X_train : (419, 233)
y_train : (419,)
Class weights: {'Mild Demented': 1.6115, 'Moderate Demented': 3.4917, 'Non-Demented': 0.3909, 'Very Mild Demented': 1.8705}


In [11]:
# Setting up cross-validation strategy

# Using 5-fold stratified CV - 10 folds would leave too few Moderate samples per fold (~3)
# 5 folds gives ~6 Moderate samples per fold, which is the minimum stable count
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

In [12]:
# Training Linear kernel SVM

param_grid_linear = {'C': [0.01, 0.1, 1, 10, 100]}

grid_linear = GridSearchCV(
    SVC(kernel='linear', class_weight=class_weight_dict, random_state=RANDOM_SEED, probability=True),
    param_grid_linear,
    cv=cv,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)
grid_linear.fit(X_train, y_train)

svm_linear = grid_linear.best_estimator_
print(f'Best params : {grid_linear.best_params_}')
print(f'CV F1 macro : {grid_linear.best_score_:.4f}')

Fitting 5 folds for each of 5 candidates, totalling 25 fits
Best params : {'C': 0.1}
CV F1 macro : 0.7581


In [13]:
# Training RBF kernel SVM

param_grid_rbf = {
    'C':     [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01]
}

grid_rbf = GridSearchCV(
    SVC(kernel='rbf', class_weight=class_weight_dict, random_state=RANDOM_SEED, probability=True),
    param_grid_rbf,
    cv=cv,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)
grid_rbf.fit(X_train, y_train)

svm_rbf = grid_rbf.best_estimator_
print(f'Best params : {grid_rbf.best_params_}')
print(f'CV F1 macro : {grid_rbf.best_score_:.4f}')

Fitting 5 folds for each of 16 candidates, totalling 80 fits
Best params : {'C': 100, 'gamma': 'scale'}
CV F1 macro : 0.7732


In [14]:
# Saving trained models

with open(os.path.join(MODELS_DIR, 'svm_linear.pkl'), 'wb') as f:
    pickle.dump(svm_linear, f)

with open(os.path.join(MODELS_DIR, 'svm_rbf.pkl'), 'wb') as f:
    pickle.dump(svm_rbf, f)

print('Saved svm_linear.pkl and svm_rbf.pkl')

Saved svm_linear.pkl and svm_rbf.pkl


In [15]:
# Saving cross-validation scores

cv_rows = []

for params, score in zip(
    grid_linear.cv_results_['params'],
    grid_linear.cv_results_['mean_test_score']
):
    cv_rows.append({'model': 'Linear', 'params': str(params), 'mean_f1_macro': round(score, 4)})

for params, score in zip(
    grid_rbf.cv_results_['params'],
    grid_rbf.cv_results_['mean_test_score']
):
    cv_rows.append({'model': 'RBF', 'params': str(params), 'mean_f1_macro': round(score, 4)})

df_cv = pd.DataFrame(cv_rows)
df_cv.to_csv(os.path.join(METRICS_DIR, 'cv_scores.csv'), index=False)

print('CV scores summary:')
print(df_cv.groupby('model')['mean_f1_macro'].max().rename('best_f1_macro'))

CV scores summary:
model
Linear    0.7581
RBF       0.7732
Name: best_f1_macro, dtype: float64
